In [1]:
from pathlib import Path
import ast
import numpy as np
import pandas as pd
import rasterio

# Optional but recommended for accurate cell areas on EPSG:4326 grids
from pyproj import Geod

In [2]:
# -----------------------------
# Paths (EDIT THESE)
# -----------------------------
CSV_UPSTREAM = Path(r"C:\Users\SID-DRW\Downloads\centroids_upstream_basins_merged.csv")  # input CSV with upstream_hybas_ids column
GRIDINFO_TIF = Path(r"C:\Users\SID-DRW\Downloads\CHIRPS_Brahmaputra_gridinfo.tif")

OUT_DIR = Path(r"C:\Users\SID-DRW\Downloads\upstream_masks")  # output folder for per-centroid masks
OUT_DIR.mkdir(parents=True, exist_ok=True)

OUT_SUMMARY_CSV = OUT_DIR / "centroids_upstream_area_summary.csv"


# -----------------------------
# Helpers
# -----------------------------
def parse_upstream_list(val):
    """
    upstream_hybas_ids in your CSV is typically stored as a string like:
      "[4070..., 4070..., ...]"
    This parses it robustly into a np.int64 array.
    """
    if isinstance(val, (list, tuple, np.ndarray)):
        return np.asarray(val, dtype=np.int64)
    if isinstance(val, str):
        # handles python-list string representation
        return np.asarray(ast.literal_eval(val), dtype=np.int64)
    raise TypeError(f"Unsupported upstream_hybas_ids type: {type(val)}")


def pixel_area_by_row_wgs84(transform, height):
    """
    Compute WGS84 geodesic area (m^2) for one pixel per row (constant across columns
    for a regular lat/lon grid). Returns array length = height.
    """
    geod = Geod(ellps="WGS84")

    dx = transform.a          # pixel width in degrees (positive)
    dy = transform.e          # pixel height in degrees (negative for north-up)
    x0 = transform.c          # top-left lon
    y0 = transform.f          # top-left lat

    lon_left = x0
    lon_right = x0 + dx

    area_row = np.zeros(height, dtype=np.float64)

    for r in range(height):
        lat_top = y0 + r * dy
        lat_bot = y0 + (r + 1) * dy

        lons = [lon_left, lon_right, lon_right, lon_left]
        lats = [lat_top,  lat_top,   lat_bot,    lat_bot]

        area, _perim = geod.polygon_area_perimeter(lons, lats)
        area_row[r] = abs(area)  # m^2 (positive)

    return area_row


# -----------------------------
# 1) Load inputs
# -----------------------------
centroids = pd.read_csv(CSV_UPSTREAM)
required_cols = {"centroid_x", "centroid_y", "r", "centroid_hybas_id", "upstream_hybas_ids"}
missing = required_cols - set(centroids.columns)
if missing:
    raise ValueError(f"CSV missing required columns: {missing}")

with rasterio.open(GRIDINFO_TIF) as ds:
    hybas_grid = ds.read(1)  # float grid with NaNs outside basin
    meta = ds.meta.copy()
    transform = ds.transform
    height, width = ds.height, ds.width

# valid pixels are those with a HYBAS_ID number (not NaN)
valid = np.isfinite(hybas_grid)

# convert HYBAS grid to int64 only where valid
hybas_int = np.zeros((height, width), dtype=np.int64)
hybas_int[valid] = hybas_grid[valid].astype(np.int64)

# Precompute pixel area per row (m^2) for fast area sums
area_row_m2 = pixel_area_by_row_wgs84(transform, height)


# -----------------------------
# 2) For each centroid: mask + area + write TIFF
# -----------------------------
summary_rows = []

meta_out = meta.copy()
meta_out.update(
    dtype="uint8",
    count=1,
    nodata=0,
    compress="deflate",  # or "lzw"
)

for i, row in centroids.iterrows():
    r_label = str(row["r"]) if pd.notna(row["r"]) else f"centroid_{i}"
    upstream_ids = parse_upstream_list(row["upstream_hybas_ids"])

    # Build mask: 1 where (pixel HYBAS_ID is in upstream list), else 0
    mask = np.zeros((height, width), dtype=np.uint8)

    # Only compute membership on valid pixels (faster + avoids NaN issues)
    membership = np.isin(hybas_int[valid], upstream_ids)
    mask_valid = mask[valid]
    mask_valid[:] = membership.astype(np.uint8)
    mask[valid] = mask_valid

    # Area: sum per-row count * area_per_pixel_row
    counts_per_row = mask.sum(axis=1).astype(np.float64)
    upstream_area_m2 = float((counts_per_row * area_row_m2).sum())
    upstream_area_km2 = upstream_area_m2 / 1e6

    # Write mask GeoTIFF
    out_tif = OUT_DIR / f"upstream_mask_{r_label}.tif"
    with rasterio.open(out_tif, "w", **meta_out) as dst:
        dst.write(mask, 1)

    summary_rows.append(
        {
            "r": r_label,
            "centroid_x": float(row["centroid_x"]),
            "centroid_y": float(row["centroid_y"]),
            "centroid_hybas_id": int(row["centroid_hybas_id"]),
            "n_upstream_basins": int(len(upstream_ids)),
            "upstream_pixel_count": int(mask.sum()),
            "upstream_area_km2": upstream_area_km2,
            "mask_tif": str(out_tif),
        }
    )

summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(OUT_SUMMARY_CSV, index=False)

print("Wrote masks to:", OUT_DIR)
print("Wrote summary CSV:", OUT_SUMMARY_CSV)
print(summary_df.head())


Wrote masks to: C:\Users\SID-DRW\Downloads\upstream_masks
Wrote summary CSV: C:\Users\SID-DRW\Downloads\upstream_masks\centroids_upstream_area_summary.csv
    r  centroid_x  centroid_y  centroid_hybas_id  n_upstream_basins  \
0  r2   89.677827   25.244097         4070928420                226   
1  r3   89.697230   25.354590         4070928420                226   
2  r4   89.714478   25.481252         4070910920                222   
3  r1   89.655459   25.172950         4070928420                226   
4  r9   90.299281   26.158492         4070892310                208   

   upstream_pixel_count  upstream_area_km2  \
0                 19134      518994.120952   
1                 19134      518994.120952   
2                 18572      503537.873018   
3                 19134      518994.120952   
4                 17322      469164.286455   

                                            mask_tif  
0  C:\Users\SID-DRW\Downloads\upstream_masks\upst...  
1  C:\Users\SID-DRW\Downloads\u